# Character Stats Agent 테스트 (Production Level)

캐릭터 게임 데이터 추출 에이전트 테스트 노트북

## 역할: "Game Designer" (게임 디자이너)
- 능력치 (stats: strength, dexterity, intelligence, constitution, level, skills)
- 상태 (state: hp, mp, status_effects) - Hot Data
- 전투 (combat: base_attack, total_attack, source) - Base vs Total 분리
- 경제 (economy: gold)
- 사회성 (social: rank, influence)

> **⚠️ 핵심 검증**: 게임 데이터가 없는 텍스트에서 `has_game_data=false`여야 함

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
# 게임 데이터가 포함된 스토리
SAMPLE_STORY_WITH_STATS = """아린은 레벨 25의 검사였다. HP 450/500, MP 80/100. 그녀의 공격력은 150, 방어력은 80이었다.

카엘은 레벨 30의 흑기사. HP 600/600으로 완전한 상태였다. 그는 500 골드를 가지고 있었다."""

# 일반 소설 텍스트 (게임 데이터 없음)
SAMPLE_STORY_NO_STATS = """아린은 어두운 숲 한가운데 서 있었다. 긴 검은 머리카락을 바람에 휘날리며, 손에 쥔 은빛 검을 꼭 움켜쥐었다.

그림자 속에서 카엘이 나타났다. 전직 기사는 검은 갑옷을 입고 있었다."""

def create_base_state(story):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Stats Agent 실행 (게임 데이터 있는 경우)

In [3]:
from app.agents.extraction.character.stats import stats_extraction_node

async def test_stats_with_data():
    print("📊 Stats Agent 테스트 (게임 데이터 있음)...")
    return await stats_extraction_node(create_base_state(SAMPLE_STORY_WITH_STATS))

result_with_stats = run_async(test_stats_with_data())

if result_with_stats.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result_with_stats.get('errors', []):
        print(f"   {err}")
else:
    stats_data = result_with_stats.get('char_stats', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(stats_data)}개")
    print(f"   - 이름: {list(stats_data.keys())}")

📊 Stats Agent 테스트 (게임 데이터 있음)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['아린', '카엘']


## 2. Human-Readable 출력

In [4]:
stats_data = result_with_stats.get('char_stats', {})

print("="*70)
print("📊 Stats Data (캐릭터별 게임 데이터)")
print("="*70)

for name, data in stats_data.items():
    stats = data.get('stats', {})
    state = data.get('state', {})
    combat = data.get('combat', {})
    economy = data.get('economy', {})
    social = data.get('social', {})
    
    print(f"\n🧑 {name}")
    
    # Level & Stats
    if stats.get('level'):
        print(f"   레벨: {stats.get('level')}")
    
    # Base stats (normalized names)
    if stats.get('strength') or stats.get('dexterity'):
        print(f"   STR: {stats.get('strength', 'N/A')} | DEX: {stats.get('dexterity', 'N/A')} | INT: {stats.get('intelligence', 'N/A')} | CON: {stats.get('constitution', 'N/A')}")
    
    # HP/MP visualization
    if state.get('hp') is not None and state.get('hp_max'):
        hp_pct = state['hp'] / state['hp_max']
        hp_bar = '█' * int(hp_pct * 10) + '░' * (10 - int(hp_pct * 10))
        print(f"   HP: [{hp_bar}] {state['hp']}/{state['hp_max']}")
    
    if state.get('mp') is not None and state.get('mp_max'):
        mp_pct = state['mp'] / state['mp_max']
        mp_bar = '█' * int(mp_pct * 10) + '░' * (10 - int(mp_pct * 10))
        print(f"   MP: [{mp_bar}] {state['mp']}/{state['mp_max']}")
    
    # Combat (new field names)
    if combat.get('total_attack') or combat.get('total_defense'):
        source = combat.get('source', 'unknown')
        print(f"   ⚔️ 공격력: {combat.get('total_attack', 'N/A')} | 방어력: {combat.get('total_defense', 'N/A')} (source: {source})")
    
    # Economy
    if economy.get('gold'):
        print(f"   💰 골드: {economy.get('gold')}")
    
    if not any([stats.get('level'), state.get('hp'), combat.get('total_attack'), economy.get('gold')]):
        print("   (게임 데이터 없음)")

📊 Stats Data (캐릭터별 게임 데이터)

🧑 아린
   레벨: 25
   HP: [█████████░] 450/500
   MP: [████████░░] 80/100
   ⚔️ 공격력: 150 | 방어력: 80 (source: extracted)

🧑 카엘
   레벨: 30
   HP: [██████████] 600/600
   💰 골드: 500


## 3. Stats Agent 실행 (게임 데이터 없는 경우)

In [5]:
async def test_stats_no_data():
    print("📊 Stats Agent 테스트 (게임 데이터 없음)...")
    return await stats_extraction_node(create_base_state(SAMPLE_STORY_NO_STATS))

result_no_stats = run_async(test_stats_no_data())
stats_empty = result_no_stats.get('char_stats', {})

print(f"\n✅ 추출 완료:")
print(f"   - 캐릭터 수: {len(stats_empty)}개")

# Check if mostly empty (no hallucination)
has_hallucination = False
for name, data in stats_empty.items():
    if (data.get('stats', {}).get('level') or 
        data.get('state', {}).get('hp') or
        data.get('combat', {}).get('total_attack')):
        has_hallucination = True
        print(f"   ⚠️ {name}: 게임 데이터 환각(Hallucination)됨")

if not has_hallucination:
    print("   ✅ 환각 없음 - 정상")

📊 Stats Agent 테스트 (게임 데이터 없음)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   ✅ 환각 없음 - 정상


## 4. Full JSON 출력

In [6]:
print("="*70)
print("📄 Full JSON Output (게임 데이터 있는 경우)")
print("="*70)
stats_data = result_with_stats.get('char_stats', {})
if stats_data:
    print(json.dumps(stats_data, ensure_ascii=False, indent=2))
else:
    print("{}")

📄 Full JSON Output (게임 데이터 있는 경우)
{
  "아린": {
    "name": "아린",
    "stats": {
      "strength": null,
      "dexterity": null,
      "intelligence": null,
      "constitution": null,
      "level": 25,
      "exp": null,
      "skills": []
    },
    "state": {
      "hp": 450,
      "hp_max": 500,
      "mp": 80,
      "mp_max": 100,
      "status_effects": []
    },
    "combat": {
      "base_attack": null,
      "base_defense": null,
      "total_attack": 150,
      "total_defense": 80,
      "source": "extracted",
      "attack_range": null,
      "crit_chance": null,
      "attack_type": null
    },
    "economy": {
      "gold": null,
      "trade_status": null
    },
    "social": {
      "rank": null,
      "influence": null
    }
  },
  "카엘": {
    "name": "카엘",
    "stats": {
      "strength": null,
      "dexterity": null,
      "intelligence": null,
      "constitution": null,
      "level": 30,
      "exp": null,
      "skills": []
    },
    "state": {
      "hp": 600,


## 5. Production 체크리스트

In [7]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

stats_data = result_with_stats.get('char_stats', {})
checks = []

# 1. 캐릭터 존재 (게임 데이터 있는 텍스트)
if len(stats_data) >= 2:
    checks.append(("✅", f"{len(stats_data)} characters extracted"))
else:
    checks.append(("⚠️", f"Only {len(stats_data)} characters"))

if stats_data:
    # 2. Level 추출
    has_level = any(data.get('stats', {}).get('level') for data in stats_data.values())
    if has_level:
        checks.append(("✅", "Level extracted"))
    else:
        checks.append(("❌", "Level not extracted (expected for game text)"))
    
    # 3. HP 추출
    has_hp = any(data.get('state', {}).get('hp') is not None for data in stats_data.values())
    if has_hp:
        checks.append(("✅", "HP extracted"))
    else:
        checks.append(("❌", "HP not extracted (expected for game text)"))
    
    # 4. Combat stats (NEW field names: total_attack, total_defense)
    has_combat = any(
        data.get('combat', {}).get('total_attack') or data.get('combat', {}).get('total_defense')
        for data in stats_data.values()
    )
    if has_combat:
        checks.append(("✅", "Combat stats extracted (total_attack/total_defense)"))
    else:
        checks.append(("⚠️", "Combat stats missing"))
    
    # 5. Gold (경제)
    has_gold = any(data.get('economy', {}).get('gold') for data in stats_data.values())
    if has_gold:
        checks.append(("✅", "Gold extracted"))
    else:
        checks.append(("⚠️", "Gold not extracted"))

# 6. 환각 검사 (게임 데이터 없는 텍스트)
stats_empty = result_no_stats.get('char_stats', {})
no_hallucination = not any(
    data.get('stats', {}).get('level') or data.get('state', {}).get('hp')
    for data in stats_empty.values()
)
if no_hallucination:
    checks.append(("✅", "No hallucination on non-game text"))
else:
    checks.append(("❌", "Hallucination detected on non-game text"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
print(f"결과: {passed}/{len(checks)} checks passed")

✅ Production 체크리스트

✅ 2 characters extracted
✅ Level extracted
✅ HP extracted
✅ Combat stats extracted (total_attack/total_defense)
✅ Gold extracted
✅ No hallucination on non-game text

결과: 6/6 checks passed


## 6. 디버그 정보

In [8]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"\n[게임 데이터 있는 텍스트]")
print(f"Result keys: {result_with_stats.keys()}")
print(f"Errors: {result_with_stats.get('errors', [])}")
print(f"Completed agents: {result_with_stats.get('completed_agents', [])}")

print(f"\n[게임 데이터 없는 텍스트]")
print(f"Result keys: {result_no_stats.keys()}")
print(f"Errors: {result_no_stats.get('errors', [])}")

🔍 디버그 정보

[게임 데이터 있는 텍스트]
Result keys: dict_keys(['char_stats', 'completed_agents', 'messages'])
Errors: []
Completed agents: ['stats']

[게임 데이터 없는 텍스트]
Result keys: dict_keys(['char_stats', 'completed_agents', 'messages'])
Errors: []
